In [37]:
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import math

from utils.helpers import *

In [38]:
input_path = 'data/'
output_path = 'data/preprocessed'

x_train, x_test, y_train, train_ids, test_ids = load_csv_data(data_path='data/')

In [39]:
def classify_columns(x):
    """
    Classifies columns into numerical, symbolic, cyclic, and binary 
    based purely on the number of unique values in the NumPy array.
    """
    cyclic_indices = [2, 4, 5]  # FMONTH, IMONTH, IDAY

    cyclic_cols = list(cyclic_indices)
    symbolic_cols = []  
    binary_cols = []
    numerical_cols = []

    num_features = x.shape[1]
    
    for col in range(num_features):
        if col in cyclic_cols:
            continue

        col_data = x[:, col]
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        valid_data = col_data[mask]
        
        if len(valid_data) == 0:
            continue
            
        nunique = len(np.unique(valid_data))
        
        if nunique <= 2:
            binary_cols.append(col)
        elif 2 < nunique < 25:
            symbolic_cols.append(col)
        else:
            numerical_cols.append(col)
            
    return numerical_cols, symbolic_cols, cyclic_cols, binary_cols


def drop_high_missing_columns(x_train, x_test, threshold=0.50, missing_val=-999.0):
    """Drops columns from x_train and x_test where missing ratio in x_train > threshold."""
    is_missing = np.isnan(x_train) | (x_train == missing_val)
    missing_ratios = np.mean(is_missing, axis=0)
    
    cols_to_keep = np.where(missing_ratios <= threshold)[0]
    
    print(f"Original columns: {x_train.shape[1]}")
    print(f"Dropped {x_train.shape[1] - len(cols_to_keep)} columns with > {int(threshold * 100)}% missing values.")
    print(f"Remaining columns: {len(cols_to_keep)}")
    
    return x_train[:, cols_to_keep], x_test[:, cols_to_keep]

# 1. Drop high-missingness columns first
x_train_clean, x_test_clean = drop_high_missing_columns(x_train, x_test, threshold=0.50)

# 2. Classify remaining columns on the cleaned matrix
numerical_cols, symbolic_cols, cyclic_cols, binary_cols = classify_columns(x_train_clean)


Original columns: 321
Dropped 147 columns with > 50% missing values.
Remaining columns: 174


# 1. Variable Encoding

In [40]:
def ordinal_encoding(x, ord_cols):
    """Maps ordinal features to strictly sequential integers (0, 1, 2...)."""
    if not ord_cols:
        return x
        
    x_encoded = np.copy(x)
    for col in ord_cols:
        mask = (~np.isnan(x_encoded[:, col])) & (x_encoded[:, col] != -999.0)
        unique_vals = np.unique(x_encoded[mask, col])
        
        # Apply mapping: value -> index
        for i, val in enumerate(unique_vals):
            x_encoded[x_encoded[:, col] == val, col] = i
            
    return x_encoded


def encode_symbolic(x, sym_cols):
    """
    One-Hot Encodes categorical features.
    Original columns are removed, and new binary dummy columns are appended to the right.
    """
    if not sym_cols:
        return x
    
    # Keep columns that are NOT being one-hot encoded
    cols_to_keep = [i for i in range(x.shape[1]) if i not in sym_cols]
    encoded_arrays = [x[:, cols_to_keep]]
    
    for col in sym_cols:
        col_data = x[:, col]
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        unique_vals = np.unique(col_data[mask])
        
        # Create dummy columns for each unique category
        one_hot = np.zeros((x.shape[0], len(unique_vals)), dtype=float)
        for i, val in enumerate(unique_vals):
            one_hot[:, i] = (col_data == val).astype(float)
            
        # Preserve NaNs in the one-hot vectors if missing values weren't imputed yet
        one_hot[~mask, :] = np.nan
        
        encoded_arrays.append(one_hot)
        
    return np.concatenate(encoded_arrays, axis=1)


def encode_cyclic(x, cyclic_cols):
    """
    Transforms cyclic temporal features into sine and cosine components.
    (e.g., makes Month 12 and Month 1 geometrically close).
    """
    if not cyclic_cols:
        return x
    
    cols_to_keep = [i for i in range(x.shape[1]) if i not in cyclic_cols]
    encoded_arrays = [x[:, cols_to_keep]]
    
    for col in cyclic_cols:
        col_data = x[:, col]
        # Ignore NaNs/Missing for max_val calculation
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        max_val = np.max(col_data[mask]) if np.any(mask) else 1
        
        # Sine and Cosine transformations
        sin_col = np.sin(2 * np.pi * col_data / max_val).reshape(-1, 1)
        cos_col = np.cos(2 * np.pi * col_data / max_val).reshape(-1, 1)
        
        # Restore NaNs where data was originally missing
        sin_col[~mask] = np.nan
        cos_col[~mask] = np.nan
        
        encoded_arrays.append(sin_col)
        encoded_arrays.append(cos_col)
        
    return np.concatenate(encoded_arrays, axis=1)


def encode_binary(x, bin_cols):
    """Standardizes binary variables strictly to {0, 1}."""
    if not bin_cols:
        return x
    
    x_encoded = np.copy(x)
    for col in bin_cols:
        col_data = x_encoded[:, col]
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        unique_vals = np.unique(col_data[mask])
        
        if len(unique_vals) == 2:
            x_encoded[col_data == unique_vals[0], col] = 0
            x_encoded[col_data == unique_vals[1], col] = 1
        elif len(unique_vals) == 1:
            x_encoded[col_data == unique_vals[0], col] = 0
            
    return x_encoded

def apply_all_encodings(x, num_cols, sym_cols, cyc_cols, bin_cols):
    """Applies binary, cyclic, and symbolic encodings cleanly without index-drift errors."""
    # 1. Binary Encoding (in-place modification, keeps original column indices intact)
    x_encoded = encode_binary(x, bin_cols)
    
    # Identify unchanged columns (numerical + binary) to preserve
    cols_to_keep = [i for i in range(x.shape[1]) if i not in cyc_cols and i not in sym_cols]
    encoded_blocks = [x_encoded[:, cols_to_keep]]
    
    # 2. Cyclic Encoding (Sine / Cosine expansion)
    for col in cyc_cols:
        col_data = x_encoded[:, col]
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        max_val = np.max(col_data[mask]) if np.any(mask) else 1.0
        
        sin_col = np.sin(2 * np.pi * col_data / max_val).reshape(-1, 1)
        cos_col = np.cos(2 * np.pi * col_data / max_val).reshape(-1, 1)
        
        sin_col[~mask] = np.nan
        cos_col[~mask] = np.nan
        
        encoded_blocks.append(sin_col)
        encoded_blocks.append(cos_col)
        
    # 3. Symbolic Encoding (One-Hot dummy expansion)
    for col in sym_cols:
        col_data = x_encoded[:, col]
        mask = (~np.isnan(col_data)) & (col_data != -999.0)
        unique_vals = np.unique(col_data[mask])
        
        one_hot = np.zeros((x_encoded.shape[0], len(unique_vals)), dtype=float)
        for i, val in enumerate(unique_vals):
            one_hot[:, i] = (col_data == val).astype(float)
            
        one_hot[~mask, :] = np.nan
        encoded_blocks.append(one_hot)
        
    return np.concatenate(encoded_blocks, axis=1)

In [41]:
import numpy as np


def fit_transform_encodings(x_train, x_test, num_cols, sym_cols, cyc_cols, bin_cols):
    """Encodes features sequentially in original column order using K-1 One-Hot Encoding

    (dropping the first category) to prevent the Dummy Variable Trap.
    """
    x_tr = np.copy(x_train)
    x_te = np.copy(x_test)

    num_set, sym_set = set(num_cols), set(sym_cols)
    cyc_set, bin_set = set(cyc_cols), set(bin_cols)

    blocks_tr, blocks_te = [], []
    encoded_num_cols = []
    current_col_idx = 0
    num_features = x_train.shape[1]

    for col in range(num_features):
        col_tr = x_tr[:, col]
        col_te = x_te[:, col]

        # 1. Continuous Numerical Features
        if col in num_set:
            blocks_tr.append(col_tr.reshape(-1, 1))
            blocks_te.append(col_te.reshape(-1, 1))
            encoded_num_cols.append(current_col_idx)
            current_col_idx += 1

        # 2. Binary Features (1 column)
        elif col in bin_set:
            vals = np.unique(col_tr)
            bin_tr = np.copy(col_tr).reshape(-1, 1)
            bin_te = np.copy(col_te).reshape(-1, 1)

            if len(vals) >= 1:
                bin_tr[col_tr == vals[0]] = 0
                bin_te[col_te == vals[0]] = 0
                if len(vals) > 1:
                    bin_tr[col_tr == vals[1]] = 1
                    bin_te[col_te == vals[1]] = 1

            blocks_tr.append(bin_tr)
            blocks_te.append(bin_te)
            current_col_idx += 1

        # 3. Cyclic Features (2 columns: Sin & Cos)
        elif col in cyc_set:
            max_val = np.max(col_tr) if np.max(col_tr) > 0 else 1.0

            blocks_tr.extend([
                np.sin(2 * np.pi * col_tr / max_val).reshape(-1, 1),
                np.cos(2 * np.pi * col_tr / max_val).reshape(-1, 1),
            ])
            blocks_te.extend([
                np.sin(2 * np.pi * col_te / max_val).reshape(-1, 1),
                np.cos(2 * np.pi * col_te / max_val).reshape(-1, 1),
            ])
            current_col_idx += 2

        # 4. Symbolic Features: K-1 One-Hot Encoding (Drop first category!)
        elif col in sym_set:
            categories = np.unique(col_tr)

            # Drop category 0 to avoid the Dummy Variable Trap
            categories_to_encode = (
                categories[1:] if len(categories) > 1 else categories
            )

            oh_tr = np.zeros(
                (x_train.shape[0], len(categories_to_encode)), dtype=float
            )
            for i, val in enumerate(categories_to_encode):
                oh_tr[:, i] = (col_tr == val).astype(float)
            blocks_tr.append(oh_tr)

            oh_te = np.zeros(
                (x_test.shape[0], len(categories_to_encode)), dtype=float
            )
            for i, val in enumerate(categories_to_encode):
                oh_te[:, i] = (col_te == val).astype(float)
            blocks_te.append(oh_te)

            current_col_idx += len(categories_to_encode)

    x_tr_out = np.concatenate(blocks_tr, axis=1)
    x_te_out = np.concatenate(blocks_te, axis=1)

    return x_tr_out, x_te_out, encoded_num_cols

In [42]:
# Transform train and test matrices together using Fit-Transform pattern
x_train_encoded, x_test_encoded, encoded_num_cols = fit_transform_encodings(
    x_train_clean, x_test_clean, numerical_cols, symbolic_cols, cyclic_cols, binary_cols
)

print("Original x_train shape:", x_train.shape)
print("Encoded x_train shape: ", x_train_encoded.shape)
print("Encoded x_test shape:  ", x_test_encoded.shape)

Original x_train shape: (328135, 321)
Encoded x_train shape:  (328135, 529)
Encoded x_test shape:   (109379, 529)


# 2. Missing Values Imputation

In [43]:
def impute_missing_values(x_train_enc, x_test_enc, missing_val=-999.0):
    x_tr = np.copy(x_train_enc)
    x_te = np.copy(x_test_enc)
    
    x_tr[x_tr == missing_val] = np.nan
    x_te[x_te == missing_val] = np.nan
    
    for col in range(x_tr.shape[1]):
        col_tr = x_tr[:, col]
        mask_tr = ~np.isnan(col_tr)
        
        fill_val = np.nanmedian(col_tr) if np.any(mask_tr) else 0.0
        
        x_tr[np.isnan(x_tr[:, col]), col] = fill_val
        x_te[np.isnan(x_te[:, col]), col] = fill_val
        
    return x_tr, x_te

x_train_imp, x_test_imp = impute_missing_values(x_train_encoded, x_test_encoded)

# 3. Outliers

In [44]:
def handle_outliers(x, num_cols):
    """Clips outliers in numerical columns using the Interquartile Range (IQR)."""
    x_out = np.copy(x)
    
    for col in num_cols:
        q1 = np.percentile(x_out[:, col], 25)
        q3 = np.percentile(x_out[:, col], 75)
        iqr = q3 - q1
        
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Clip values to bounds
        x_out[:, col] = np.clip(x_out[:, col], lower_bound, upper_bound)
        
    return x_out

x_train_no_out = handle_outliers(x_train_imp, encoded_num_cols)
x_test_no_out = handle_outliers(x_test_imp, encoded_num_cols)

# 4. Scaling

In [45]:
def standardize(x_train, x_test, num_cols):
    """Standardizes strictly the numerical variables (Z-score normalization)."""
    x_train_scaled = np.copy(x_train)
    x_test_scaled = np.copy(x_test)
    
    # We DO NOT scale binary or categorical one-hot encoded columns
    for col in num_cols:
        mean = np.mean(x_train[:, col])
        std = np.std(x_train[:, col])
        
        if std == 0:
            std = 1e-8 # Prevent division by zero
            
        x_train_scaled[:, col] = (x_train[:, col] - mean) / std
        x_test_scaled[:, col] = (x_test[:, col] - mean) / std
        
    return x_train_scaled, x_test_scaled

x_train_scaled, x_test_scaled = standardize(x_train_no_out, x_test_no_out, encoded_num_cols)

# 5. Data Balancing

In [46]:
# def balance_classes(x, y):
#     """Undersamples the majority class to balance the dataset."""
#     classes, counts = np.unique(y, return_counts=True)
#     if len(classes) < 2:
#         return x, y
        
#     majority_class = classes[np.argmax(counts)]
#     minority_class = classes[np.argmin(counts)]
    
#     idx_maj = np.where(y == majority_class)[0]
#     idx_min = np.where(y == minority_class)[0]
    
#     np.random.seed(42)
#     idx_maj_sampled = np.random.choice(idx_maj, size=len(idx_min), replace=False)
    
#     balanced_idx = np.concatenate([idx_min, idx_maj_sampled])
#     np.random.shuffle(balanced_idx)
    
#     return x[balanced_idx], y[balanced_idx]

# x_train_bal, y_train_bal = balance_classes(x_train_scaled, y_train)

# Data inspection and saving 

In [47]:
import os
os.makedirs(output_path, exist_ok=True)

# Inspect shapes
print("--- Final Matrix Shapes ---")
print("x_train:", x_train_scaled.shape)
print("x_test: ", x_test_scaled.shape)
print("NaNs present:", np.isnan(x_train_scaled).any())  # Should print False

np.save(os.path.join(output_path, 'x_train_preprocessed.npy'), x_train_scaled)
np.save(os.path.join(output_path, 'x_test_preprocessed.npy'), x_test_scaled)

print("Preprocessing complete.")

--- Final Matrix Shapes ---
x_train: (328135, 529)
x_test:  (109379, 529)
NaNs present: False
Preprocessing complete.
